# Coffee17 MVFD-SBN Causal Ablation — AUXCE-SBN Control\n\nTujuan notebook ini **bukan mencari metode baru**, tetapi menguji apakah gain MVFD-SBN benar-benar membutuhkan feature-distillation term.\n\nSemua komponen MVFD-SBN dipertahankan kecuali `lambda_feat = 0`. Jadi objective control hanya `CE_R + 0.05(CE_C+CE_F+CE_W)`. Teacher feature tetap dihitung hanya untuk diagnostic dan tidak memberi gradient.\n\n**Sebelum Run All:** cukup Add Input Coffee17 original, aktifkan GPU dan Internet. Jangan attach checkpoint lama.\n\nOuter test tidak disentuh.\n

In [ ]:
# Coffee17 MVFD-SBN causal ablation: AUXCE-SBN control
SCIENTIFIC_CODE_COMMIT = "d449f6caf2468e3ec99fe76a806e763094939a4f"

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-mvfd-ablation-code"
PROJECT = WORK / "coffee17-mvfd-sbn-ablation-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Kaggle input conflict: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Optional resume from a previous saved output of THIS notebook.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-mvfd-sbn-ablation-project")
    if p.is_dir()
):
    print("MERGE PRIOR ABLATION OUTPUT:", prior)
    merge_tree_exact(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact scientific code.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.engine.shared_multiview import validation_identity_label_sha256

R0_REFERENCE = REPO / "configs/shared_multiview/sbn_reference_v1.json"
MVFD_REFERENCE = REPO / "configs/mvfd_sbn/mvfd_result_reference_v1.json"
CONFIG = REPO / "configs/mvfd_sbn/AUXCE_SBN_CONTROL.yaml"
OUT = PROJECT / "experiments/coffee17-mvfd-sbn-ablation-v1"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 folds from original dataset only.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")
by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus memuat Coffee17 original valid saja "
        f"(expected 979/17, observed {raw_count}/{len(by_class)})."
    )

ARCHIVE = WORK / "coffee17_mvfd_ablation_original.zip"
PROV_LOCAL = WORK / "coffee17_mvfd_ablation_provenance"
CANONICAL = WORK / "coffee17_mvfd_ablation_original_v1"
FOLDS_LOCAL = WORK / "coffee17_mvfd_ablation_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

r0_reference = json.loads(R0_REFERENCE.read_text(encoding="utf-8"))
mvfd_reference = json.loads(MVFD_REFERENCE.read_text(encoding="utf-8"))
print("R0 REFERENCE SHA  :", sha256_file(R0_REFERENCE))
print("MVFD ARTIFACT SHA :", mvfd_reference["source_artifact_sha256"])
print("EXPECTED INIT      :", r0_reference["expected_initial_model_state_sha256"])

# ------------------------------------------------------------
# 3. Run only lambda_feat=0 causal control, five folds.
# ------------------------------------------------------------
print("\n=== 2/4 AUXCE-SBN CONTROL FIVE FOLDS ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_mvfd_ablation_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    count, val_sha = validation_identity_label_sha256(DEV)
    r0_expected = r0_reference["folds"][f"fold_{fold}"]
    mvfd_expected = mvfd_reference["folds"][f"fold_{fold}"]
    print("VAL COUNT:", count)
    print("VAL SHA  :", val_sha)
    if (
        count != r0_expected["count"]
        or count != mvfd_expected["count"]
        or val_sha != r0_expected["identity_label_sha256"]
        or val_sha != mvfd_expected["identity_label_sha256"]
    ):
        raise RuntimeError(f"Fold {fold}: frozen-reference mismatch.")
    print("VALIDATION REFERENCE: PASS")

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_auxce_sbn_control",
        "--config", CONFIG,
        "--r0-reference", R0_REFERENCE,
        "--mvfd-reference", MVFD_REFERENCE,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"AUXCE_SBN_CONTROL_fold{fold}.log")

    result = json.loads(
        (
            OUT
            / "AUXCE_SBN_CONTROL"
            / f"fold_{fold}"
            / "seed42"
            / "result.json"
        ).read_text(encoding="utf-8")
    )
    print(
        f"FOLD {fold} | "
        f"CONTROL Macro={result['metrics']['macro_f1']:.4f} | "
        f"vs R0={result['delta_vs_r0_control']['macro_f1']:+.4f} | "
        f"vs MVFD={result['delta_vs_mvfd_sbn']['macro_f1']:+.4f} | "
        f"weighted feature={result['best_epoch_diagnostics']['weighted_feature_loss']:.6f}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Summary + compact analysis package.
# ------------------------------------------------------------
print("\n=== 3/4 SUMMARY ===")
SUMMARY = PROJECT / "analysis" / "mvfd_sbn_causal_ablation_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_mvfd_sbn_ablation_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]

print("\n=== CAUSAL RESULT ===")
print("R0 CONTROL Macro-F1 :", f"{agg['R0_CONTROL']['macro_f1']['mean']:.4%}")
print("AUXCE control Macro :", f"{agg['AUXCE_SBN_CONTROL']['macro_f1']['mean']:.4%}")
print("MVFD-SBN Macro      :", f"{agg['MVFD_SBN']['macro_f1']['mean']:.4%}")
print(
    "Control - R0       :",
    f"{agg['CONTROL_MINUS_R0']['macro_f1']['mean']:+.4%}",
)
print(
    "Control - MVFD     :",
    f"{agg['CONTROL_MINUS_MVFD']['macro_f1']['mean']:+.4%}",
)
print("MVFD beats control :", summary["mvfd_beats_control_macro_folds"], "/ 5")
print("Weighted feat loss :", f"{agg['BEST_WEIGHTED_FEATURE_LOSS']['mean']:.8f}")

print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")
PKG = WORK / "mvfd-sbn-ablation-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")
for fold in range(1, 6):
    src = OUT / "AUXCE_SBN_CONTROL" / f"fold_{fold}" / "seed42"
    dst = PKG / "AUXCE_SBN_CONTROL" / f"fold_{fold}"
    dst.mkdir(parents=True, exist_ok=True)
    for name in ("result.json", "run_contract.json", "history.json", "run_config.yaml"):
        p = src / name
        if p.is_file():
            shutil.copy2(p, dst / name)
    if (src / "validation").is_dir():
        shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(R0_REFERENCE, PKG / "matched_r0_reference.json")
shutil.copy2(MVFD_REFERENCE, PKG / "frozen_mvfd_reference.json")
shutil.copy2(CONFIG, PKG / "AUXCE_SBN_CONTROL.yaml")
protocol = REPO / "docs/protocols/MVFD_SBN_CAUSAL_ABLATION_V1.md"
if protocol.is_file():
    shutil.copy2(protocol, PKG / "MVFD_SBN_CAUSAL_ABLATION_V1.md")

zip_path = Path(
    shutil.make_archive(
        str(WORK / "mvfd-sbn-ablation-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Tidak ada outer-test inference dan tidak membutuhkan checkpoint lama.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
